In [1]:
# ============================================
# BREAST CANCER CLASSIFICATION ASSIGNMENT
# Baseline + GridSearchCV + RandomizedSearchCV + Ensemble
# ============================================

# 1. Import libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV,
    RandomizedSearchCV
)

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score


# ============================================
# 2. Load dataset
# ============================================

df = pd.read_csv("/content/Breast Cancer Detection.zip")

print(df.head())
print(df.shape)
print(df.info())


# ============================================
# 3. Remove unnecessary columns
# ============================================

df = df.drop(columns=["id"], errors="ignore")
df = df.drop(columns=["Unnamed: 32"], errors="ignore")


# ============================================
# 4. Encode target column
# ============================================

le = LabelEncoder()

df["diagnosis"] = le.fit_transform(df["diagnosis"])

print("Classes:", le.classes_)


# ============================================
# 5. Separate features and target
# ============================================

X = df.drop("diagnosis", axis=1)
y = df["diagnosis"]

print("X shape:", X.shape)
print("y shape:", y.shape)


# ============================================
# 6. Train-test split
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)


# ============================================
# 7. Baseline Logistic Regression
# ============================================

baseline_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000))
])


# 5-fold Cross Validation

baseline_cv_scores = cross_val_score(
    baseline_model,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy"
)

baseline_cv_accuracy = baseline_cv_scores.mean()

print("\nBaseline CV Scores:")
print(baseline_cv_scores)

print("Baseline Mean CV Accuracy:")
print(baseline_cv_accuracy)


# Train baseline model

baseline_model.fit(X_train, y_train)

baseline_pred = baseline_model.predict(X_test)

baseline_test_accuracy = accuracy_score(
    y_test,
    baseline_pred
)

print("Baseline Test Accuracy:")
print(baseline_test_accuracy)


# ============================================
# 8. GridSearchCV
# ============================================

grid_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000))
])


grid_params = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__solver": ["liblinear", "lbfgs"]
}


grid_search = GridSearchCV(
    estimator=grid_model,
    param_grid=grid_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)


grid_search.fit(X_train, y_train)


print("\nGridSearchCV Best Parameters:")
print(grid_search.best_params_)

print("GridSearchCV Best CV Accuracy:")
print(grid_search.best_score_)


grid_pred = grid_search.best_estimator_.predict(X_test)

grid_test_accuracy = accuracy_score(
    y_test,
    grid_pred
)

print("GridSearchCV Test Accuracy:")
print(grid_test_accuracy)


# ============================================
# 9. RandomizedSearchCV
# ============================================

random_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000))
])


random_params = {
    "model__C": np.logspace(-4, 4, 50),
    "model__solver": ["liblinear", "lbfgs"]
}


random_search = RandomizedSearchCV(
    estimator=random_model,
    param_distributions=random_params,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)


random_search.fit(X_train, y_train)


print("\nRandomizedSearchCV Best Parameters:")
print(random_search.best_params_)

print("RandomizedSearchCV Best CV Accuracy:")
print(random_search.best_score_)


random_pred = random_search.best_estimator_.predict(X_test)

random_test_accuracy = accuracy_score(
    y_test,
    random_pred
)

print("RandomizedSearchCV Test Accuracy:")
print(random_test_accuracy)


# ============================================
# 10. Ensemble Model - Random Forest
# ============================================

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)


rf_cv_scores = cross_val_score(
    rf,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy"
)

rf_cv_accuracy = rf_cv_scores.mean()

print("\nRandom Forest CV Scores:")
print(rf_cv_scores)

print("Random Forest Mean CV Accuracy:")
print(rf_cv_accuracy)


rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

rf_test_accuracy = accuracy_score(
    y_test,
    rf_pred
)

print("Random Forest Test Accuracy:")
print(rf_test_accuracy)


# ============================================
# 11. Final Results Table
# ============================================

results = pd.DataFrame({
    "Model": [
        "Baseline Logistic Regression",
        "GridSearchCV Logistic Regression",
        "RandomizedSearchCV Logistic Regression",
        "Random Forest"
    ],

    "CV Accuracy": [
        baseline_cv_accuracy,
        grid_search.best_score_,
        random_search.best_score_,
        rf_cv_accuracy
    ],

    "Test Accuracy": [
        baseline_test_accuracy,
        grid_test_accuracy,
        random_test_accuracy,
        rf_test_accuracy
    ],

    "Best Params": [
        "Default",
        str(grid_search.best_params_),
        str(random_search.best_params_),
        "n_estimators=200"
    ]
})


results["CV Accuracy"] = results["CV Accuracy"].round(4)
results["Test Accuracy"] = results["Test Accuracy"].round(4)

print("\nFinal Results Table:")
print(results)

results

         id diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0    842302         M        17.99         10.38          122.80     1001.0   
1    842517         M        20.57         17.77          132.90     1326.0   
2  84300903         M        19.69         21.25          130.00     1203.0   
3  84348301         M        11.42         20.38           77.58      386.1   
4  84358402         M        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   ...  texture_worst  perimeter_worst  area_worst  smoothness

,Model,CV Accuracy,Test Accuracy,Best Params
0,Baseline Logistic Regression,0.9714,0.9649,Default
1,GridSearchCV Logistic Regression,0.9758,0.9825,"{'model__C': 0.1, 'model__solver': 'liblinear'}"
2,RandomizedSearchCV Logistic Regression,0.9758,0.9737,"{'model__solver': 'lbfgs', 'model__C': np.floa..."
3,Random Forest,0.9582,0.9649,n_estimators=200


I used Logistic Regression as the baseline model and evaluated it using 5-fold cross-validation. Then I tuned the same model using GridSearchCV and RandomizedSearchCV to find better hyperparameters. Finally, I trained a Random Forest ensemble model and compared all models using CV Accuracy and Test Accuracy in a single results table.